# Workshop 3 · 04 — Data Science auf unstrukturierten Daten (Aufgabe)

**Ziel:** Aus unstrukturiertem Text numerische Repräsentationen (**Embeddings**) erzeugen und darauf klassische Data-Science-Verfahren anwenden:
1. **Clustering** – thematisch verwandte Meldungen automatisch gruppieren.
2. **Anomalie-Erkennung** – auffällige Meldungen finden (IsolationForest auf Embeddings).
3. **Similarity-Suche** – „ähnliche Schäden finden" über semantische Ähnlichkeit statt Stichwortsuche.

**Ausgangslage:** Tabelle `asset_clean` aus Notebook 01.

> Zentrales Prinzip: **Unstrukturierte Texte → Vektoren (`ai.embed`) → klassisches ML.**

## Schritt 0 — Dokument bauen und embedden

**Aufgabe:**
1. Baue je Meldung ein "Dokument" aus `komponente`, `freitext_meldung`, `kunden_kommentar` (z.B. mit `concat_ws`).
2. Erzeuge mit `ai.embed(input_col='doc', output_col='vector')` einen Vektor pro Meldung.
3. Hole die relevanten Spalten inkl. `vector` mit `.toPandas()` in ein DataFrame `pdf`.

In [ ]:
# This code uses AI. Always review output for mistakes.
import synapse.ml.spark.aifunc as aifunc
from pyspark.sql.functions import concat_ws, col

base = spark.table('asset_clean')

# TODO 1: Baue je Meldung eine 'doc'-Spalte aus komponente + freitext_meldung + kunden_kommentar.
# TODO 2: Erzeuge mit ai.embed(...) einen Vektor pro Meldung (output_col='vector').
# TODO 3: Hole meldung_id, depot_norm, komponente, schweregrad, freitext_meldung, vector nach pandas -> pdf.
pdf = ...

## 1) Clustering — Themen automatisch finden

**Aufgabe:** Staple die Vektoren zu einer Matrix `X` und wende `KMeans` (z.B. `k=4`) darauf an. Schreibe das Cluster-Label in `pdf['cluster']` und zeige die Cluster-Größen.

*Hinweis:* `np.vstack(...)` zum Stapeln, `KMeans(n_clusters=k).fit_predict(X)`.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

# TODO 1: Vektoren aus pdf['vector'] zu einer Matrix X stapeln.
X = ...

# TODO 2: Mit KMeans (z.B. k=4) clustern und Label nach pdf['cluster'] schreiben.
# TODO 3: Cluster-Groessen anzeigen.

### (Bonus) Cluster automatisch benennen
**Aufgabe:** Erzeuge je Cluster ein prägnantes Themen-Label aus ein paar Beispiel-Freitexten — mit der pandas-AI-Function `df.ai.generate_response(prompt=..., is_prompt_template=True)`. Schreibe das Label nach `pdf['thema']`.

In [ ]:
# This code uses AI. Always review output for mistakes.
import synapse.ml.aifunc as aifunc_pd
import pandas as pd

# TODO (Bonus): Je Cluster ein Themen-Label aus Beispiel-Freitexten mit der pandas-AI-Function erzeugen
#               und nach pdf['thema'] schreiben.

## 2) Anomalie-Erkennung — ungewöhnliche Meldungen

**Aufgabe:** Wende `IsolationForest` (z.B. `contamination=0.15`) auf `X` an und markiere Anomalien in `pdf['anomalie']`. Zeige die auffälligen Meldungen.

*Hinweis:* `fit_predict(X) == -1` markiert Anomalien.

In [ ]:
from sklearn.ensemble import IsolationForest

# TODO: IsolationForest auf X anwenden, Anomalien nach pdf['anomalie'] schreiben und anzeigen.

## 3) Similarity-Suche — „finde ähnliche Schäden"

**Aufgabe:** Embedde eine Freitext-Anfrage und vergleiche sie per **Cosine-Ähnlichkeit** mit allen Meldungen (Matrix `X`). Gib die Top-Treffer zurück.

*Hinweis:* eine kleine `embed_query(q)`-Hilfsfunktion, dann `X @ qv / (||X|| * ||qv||)` und `np.argsort`.

In [ ]:
# This code uses AI. Always review output for mistakes.

# TODO 1: embed_query(q) -> embeddet eine Anfrage zu einem Vektor (ai.embed auf einem 1-Zeilen-DataFrame).
# TODO 2: find_similar(q, top_k=3) -> Cosine-Aehnlichkeit gegen X, Top-k Meldungen zurueckgeben.
# TODO 3: Teste mit einer Beispiel-Anfrage.

## Schritt 4 — Ergebnisse persistieren
**Aufgabe:** Speichere die angereicherten Meldungen (`meldung_id`, `depot_norm`, `komponente`, `schweregrad`, `cluster`, `thema`, `anomalie`) als Tabelle `meldung_ml` — Grundlage für Power BI / Activator.

In [ ]:
# TODO: Aus pdf einen Spark-DataFrame bauen (relevante Spalten) und als Tabelle 'meldung_ml' speichern.

### Ergebnis-Check
Aus Text wurden Vektoren (`ai.embed`), darauf liefen **Clustering**, **Anomalie-Erkennung** und **semantische Suche** — Methoden, die ohne Embeddings nicht möglich wären. Ergebnis `meldung_ml`.

> **Bei großen Korpora:** Vektoren in **Eventhouse/KQL** legen und mit `series_cosine_similarity()` skalierend suchen statt im Treiber.